In [2]:
import pandas as pd
import numpy as np

# =============================================================================
# CONFIGURATION
# =============================================================================
FILE_PATH = "/home/ajarrah/PhD_Thesis/chapter_4/code_final/2_mz_synced_isotope_80_matching_results_new/identified_isotopes.csv"
OUTPUT_PATH = "/home/ajarrah/PhD_Thesis/chapter_4/code_final/2_mz_synced_isotope_80_matching_results_new/parent_children_hierarchy.csv"

PRECISION = 4
TOLERANCE = 0.01 

MASS_DIFFS = {
    'Isotope (M+1)': 1.0033,
    'Isotope (M+2)': 2.0067,
    'Adduct (NH4)': 17.0265,
    'Adduct (Na)': 21.982,
    'Adduct (K)': 37.9555
}

def build_strict_hierarchy(path):
    try:
        df = pd.read_csv(path)
    except FileNotFoundError:
        print(f"Error: Could not find file at {path}")
        return None

    # Get all unique m/z values and round them for consistency
    all_mzs = sorted(pd.concat([df['mz_1'], df['mz_2']]).unique())
    all_mzs = np.array([round(x, PRECISION) for x in all_mzs])
    
    rows = []
    assigned_as_child = set()

    for parent in all_mzs:
        # Skip if this MZ has already been claimed as a child by a smaller parent
        if parent in assigned_as_child:
            continue
            
        row_dict = {'Parent_MZ': parent}
        found_any_child = False
        
        # Check each specific pattern
        for i, (label, diff) in enumerate(MASS_DIFFS.items()):
            target_mz = parent + diff
            
            # Find all candidates within tolerance
            distances = np.abs(all_mzs - target_mz)
            candidates_mask = distances <= TOLERANCE
            
            if np.any(candidates_mask):
                # Get the candidate indices
                indices = np.where(candidates_mask)[0]
                # Choose the index with the minimum distance (closest match)
                best_match_idx = indices[np.argmin(distances[indices])]
                best_match = all_mzs[best_match_idx]
                
                # Assign to the specific child slot
                row_dict[f'Child_{i+1}'] = best_match
                assigned_as_child.add(best_match)
                found_any_child = True
            else:
                row_dict[f'Child_{i+1}'] = np.nan
        
        # Only keep the row if at least one child (isotope/adduct) was found
        if found_any_child:
            rows.append(row_dict)

    if not rows:
        print("No matches found based on the provided MASS_DIFFS.")
        return None
        
    final_df = pd.DataFrame(rows)
    
    # Ensure correct column ordering
    cols = ['Parent_MZ'] + [f'Child_{i+1}' for i in range(len(MASS_DIFFS))]
    return final_df[cols]

# =============================================================================
# EXECUTION
# =============================================================================
result_table = build_strict_hierarchy(FILE_PATH)

if result_table is not None:
    print(f"\nSuccessfully grouped {len(result_table)} parent-centered families.")
    print(f"Logic: Closest match within {TOLERANCE} Da selected for each pattern.")
    print(result_table.head().to_string(float_format=f"%.{PRECISION}f"))
    
    result_table.to_csv(OUTPUT_PATH, index=False, float_format=f"%.{PRECISION}f")
    print(f"\nFinal table saved to: {OUTPUT_PATH}")


Successfully grouped 136 parent-centered families.
Logic: Closest match within 0.01 Da selected for each pattern.
   Parent_MZ  Child_1  Child_2  Child_3  Child_4  Child_5
0   339.2312      NaN      NaN 356.2652      NaN      NaN
1   351.2299 352.2358 353.2464 368.2657      NaN      NaN
2   354.2528 355.2620 356.2652      NaN      NaN      NaN
3   357.2677 358.2712      NaN      NaN      NaN      NaN
4   363.1714 364.1751 365.1871      NaN      NaN      NaN

Final table saved to: /home/ajarrah/PhD_Thesis/chapter_4/code_final/2_mz_synced_isotope_80_matching_results_new/parent_children_hierarchy.csv
